environment delpher

In [1]:
import os
import json
from lxml import etree
import requests

##### Handle the alto files

In [2]:
from gliner import GLiNER

labels = ["person", "location (geographical)", "organization"]

model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
model.eval()
print("ok")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/traitlets

ok


In [3]:
def spans_to_bio(tokens, spans):
    # tokens: list[str]
    # spans: list[{"start": int, "end": int, "label": str}]

    token_dict = {}
    index = 0
    for i, token in enumerate(tokens):
        token_dict[i] = {"token": token, "offset": index, "label": "O"}
        index += len(list(token)) + 1

    # Update labels based on spans
    for span in spans:
        span_start = span["start"]
        span_end = span["end"]
        span_label = span["label"]
        
        first = True
        for token_idx in token_dict:
            token_offset = token_dict[token_idx]["offset"]
            # Check if token offset falls within span range
            if token_offset >= span_start and token_offset < span_end:
                if first:
                    token_dict[token_idx]["label"] = f"B-{span_label[:3].upper()}"
                    first = False
                else:
                    token_dict[token_idx]["label"] = f"I-{span_label[:3].upper()}"
    
    return token_dict

In [4]:
def split_long_sentences(text):
    """
    Check if any sentence exceeds 350 characters.
    If yes, break it into smaller chunks (<350) and return updated text.
    """
    # Split by periods to get sentences
    sentences = text.split('.')
    result_sentences = []
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
            
        # Check if sentence exceeds 350 characters
        if len(sentence) > 350:
            # Break into chunks
            words = sentence.split()
            current_chunk = []
            current_length = 0
            
            for word in words:
                # If adding next word exceeds 350, save current chunk
                if current_length + len(word) + 1 > 350 and current_chunk:
                    result_sentences.append(" ".join(current_chunk))
                    current_chunk = [word]
                    current_length = len(word)
                else:
                    current_chunk.append(word)
                    current_length += len(word) + 1
            
            # Add remaining words
            if current_chunk:
                result_sentences.append(" ".join(current_chunk))
        else:
            # Sentence is fine, keep as is
            result_sentences.append(sentence)
    
    # Join sentences back with periods
    return ". ".join(result_sentences) + "."

In [ ]:
def chunk_text_by_word_limit(text, word_limit=300):
    """
    Split text into chunks respecting the word limit while not breaking sentences.
    """
    sentences = text.split('.')
    chunks = []
    current_chunk = []
    current_word_count = 0
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        
        sentence_words = sentence.split()
        sentence_word_count = len(sentence_words)
        
        # If adding this sentence would exceed limit and we have content, start new chunk
        if current_word_count + sentence_word_count > word_limit and current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sentence]
            current_word_count = sentence_word_count
        else:
            current_chunk.append(sentence)
            current_word_count += sentence_word_count
    
    # Add remaining chunk
    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')
    
    return chunks

In [ ]:
words_limit = 300
def create_block_of_text(page_tree):
    ns = {
            "alto": "http://schema.ccs-gmbh.com/ALTO"
        }
    block_of_text = []
    for text_block in page_tree.findall('.//alto:TextBlock', namespaces=ns):
        id = text_block.get("ID")
        text = []
        wc_score = []
        for string in text_block.findall('.//alto:String', namespaces=ns):
            text.append(string.get("CONTENT"))
            wc_score.append(float(string.get("WC")))
        if len(text) > words_limit:
            # divide text into chunks of max 300 words while preserving sentence boundaries
            chunks = chunk_text_by_word_limit(" ".join(text), word_limit=words_limit)
            for chunk in chunks:
                block_of_text.append({
                    "id": id,
                    "text": chunk,
                    "wc_score": sum(wc_score) / len(wc_score) if wc_score else 0
                })
        else:
            block_of_text.append({
                "id": id,
                "text": " ".join(text),
                "wc_score": sum(wc_score) / len(wc_score) if wc_score else 0
            })
    return block_of_text

In [6]:
dir_path = '../../../DATA/europeans-news/alto/'
output_path = 'pred'

parser = etree.XMLParser(recover=True)

for file in os.listdir(dir_path):
    if file.endswith('.xml'):
        file_name = ''.join(os.path.splitext(file)[0].split('.')[:-1])
        file_name = file_name.replace('_', ':')

        print(f"Processing file: {file_name}")
        url= f"http://resolver.kb.nl/resolve?{file_name}"
        response = requests.get(url)
        if response.status_code != 200:
            print(f"URL: {url} is not valid.")

        page_tree = response.content
        page_tree = etree.fromstring(page_tree, parser=parser)

        with open(f"{output_path}/{file_name}.txt", "w") as f:
            f.close()
            
        blocks = create_block_of_text(page_tree)

        # chunked_blocks = []
        # for block in blocks:
        #     if len(block["text"]) > 350:
        #         chunks = split_long_text(block["text"])
        #         for chunk in chunks:
        #             chunked_blocks.append({"id": block["id"],
        #                                 "wc_score": block["wc_score"],
        #                                 "text": chunk})
                    
        # blocks = chunked_blocks if chunked_blocks else blocks

        for block in blocks:
            entities = model.predict_entities(block["text"], labels)

            tokens = block["text"].split(" ")
            token_offsets = [ (entity["start"], entity["end"]) for entity in entities]

            
            with open(f"{output_path}/{file_name}.txt", "a+") as f:
                token_labels = spans_to_bio(tokens, entities)
                
                for token_info in token_labels.values():
                    f.write(f"{token_info['token']}\tPOS\t{token_info['label']}\n")
        

Processing file: urn=ddd:000023667:mpeg21:p002:alto


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Processing file: urn=ddd:000023649:mpeg21:p002:alto


/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 798 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 611 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Processing file: urn=ddd:000023661:mpeg21:p004:alto


...

Processing file: urn=ddd:000023651:mpeg21:p001:alto


/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 615 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 599 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 664 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Processing file: urn=ddd:000015594:mpeg21:p016:alto


/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 549 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


KeyboardInterrupt: 

for single file

In [ ]:
file_path = '../../../DATA/europeans-news/alto/urn=ddd_000023667_mpeg21_p003_alto.alto.xml'
filename = os.path.basename(file_path)
file_name = ''.join(os.path.splitext(filename)[0].split('.')[:-1])
file_name = file_name.replace('_', ':')
print(file_name)

url= f"http://resolver.kb.nl/resolve?{file_name}"
response = requests.get(url)
if response.status_code != 200:
    print(f"URL: {url} is not valid.")

page_tree = response.content
page_tree = etree.fromstring(page_tree, parser=parser)

blocks = create_block_of_text(page_tree)

with open(f"{file_name}.txt", "w") as f:
    f.close()

chunked_blocks = []
for block in blocks:
    if len(block["text"]) > 350:
        chunks = split_long_text(block["text"])
        for chunk in chunks:
            chunked_blocks.append({"id": block["id"],
                                "wc_score": block["wc_score"],
                                "text": chunk})
            
blocks = chunked_blocks if chunked_blocks else blocks

for block in blocks:
    print(f"ID: {block['id']}, Text: {block['text']}, WC Score: {round(block['wc_score'], 2)}")
    entities = model.predict_entities(block["text"], labels)

    # print(entities[0])
    for entity in entities:
        print(entity["text"], "=>", entity["label"])
    print("\n\n")

    tokens = block["text"].split(" ")
    token_offsets = [ (entity["start"], entity["end"]) for entity in entities]

    # print(f"Tokens: {tokens}")
    # print(f"Token count: {len(tokens)}")
    # print(f"Entities: {entities}")
    
    with open(f"{file_name}.txt", "a+") as f:
        token_labels = spans_to_bio(tokens, entities)
        
        for token_info in token_labels.values():
            f.write(f"{token_info['token']}\tPOS\t{token_info['label']}\n")

##### Handle BIO

In [ ]:
# read file as pandas dataframe
import pandas as pd
from pandas.errors import EmptyDataError
from seqeval.metrics import f1_score, classification_report

true_path = '../../../DATA/europeans-news/bio/'
pred_path = output_path

f1_scores = []
precision_scores = []
recall_scores = []
entity_metrics = {}

for file in os.listdir(true_path):
    if file.endswith('.bio'):
        with open(os.path.join(true_path, file)) as f:
            try:
                true_df = pd.read_csv(os.path.join(true_path, file), sep=' ', header=None, on_bad_lines='skip', engine='python')
                pred_df = pd.read_csv(os.path.join(pred_path, f"{file_name}.txt"), sep='\t', header=None, on_bad_lines='skip', engine='python')

                true_tags = [tag for tag in true_df.iloc[:, 2]]
                pred_tags = [tag for tag in pred_df.iloc[:, 2]]

                if len(true_tags) != len(pred_tags):
                    print(f"Warning: true tags and predicted tags have different lengths ({len(true_tags)} vs {len(pred_tags)}). Truncating to the shorter length.")
                    min_len = min(len(true_tags), len(pred_tags))
                    true_tags = true_tags[:min_len]
                    pred_tags = pred_tags[:min_len]

                # Get F1 score
                score = f1_score([true_tags], [pred_tags])
                f1_scores.append(score)
                
                # Get classification report as dict
                report = classification_report([true_tags], [pred_tags], output_dict=True)
                
                # Store micro/macro/weighted averages
                precision_scores.append(report['micro avg']['precision'])
                recall_scores.append(report['micro avg']['recall'])
                
                # Store per-entity scores
                for key in report.keys():
                    if key not in ['micro avg', 'macro avg', 'weighted avg']:
                        if key not in entity_metrics:
                            entity_metrics[key] = {'precision': [], 'recall': [], 'f1': []}
                        entity_metrics[key]['precision'].append(report[key]['precision'])
                        entity_metrics[key]['recall'].append(report[key]['recall'])
                        entity_metrics[key]['f1'].append(report[key]['f1-score'])
                
            except EmptyDataError:
                pass

# Calculate and print average scores
if f1_scores:
    average_f1 = sum(f1_scores) / len(f1_scores)
    average_precision = sum(precision_scores) / len(precision_scores) if precision_scores else 0
    average_recall = sum(recall_scores) / len(recall_scores) if recall_scores else 0
    
    print("=" * 50)
    print("OVERALL AVERAGE METRICS")
    print("=" * 50)
    print(f"Average F1 Score: {average_f1:.4f}")
    print(f"Average Precision: {average_precision:.4f}")
    print(f"Average Recall: {average_recall:.4f}")
    print(f"Total files processed: {len(f1_scores)}")
    
    print("\n" + "=" * 50)
    print("PER-ENTITY AVERAGE METRICS")
    print("=" * 50)
    for entity_type, scores in sorted(entity_metrics.items()):
        if scores['f1']:
            avg_precision = sum(scores['precision']) / len(scores['precision'])
            avg_recall = sum(scores['recall']) / len(scores['recall'])
            avg_f1 = sum(scores['f1']) / len(scores['f1'])
            print(f"{entity_type}:")
            print(f"  Precision: {avg_precision:.4f}")
            print(f"  Recall: {avg_recall:.4f}")
            print(f"  F1-Score: {avg_f1:.4f}")
else:
    print("No scores calculated")

In [ ]:
import pandas as pd
true_path = '../../../DATA/europeans-news/bio/'
pred_path = "."

true_df = pd.read_csv(os.path.join(true_path, f"{file_name}".replace(':', '_')+'.alto.xml.bio'), sep=' ', header=None, on_bad_lines='skip', engine='python')
pred_df = pd.read_csv(os.path.join(pred_path, f"{file_name}.txt"), sep='\t', header=None, on_bad_lines='skip', engine='python')

true_tags = [tag for tag in true_df.iloc[:, 2]]
pred_tags = [tag for tag in pred_df.iloc[:, 2]]

In [ ]:
from seqeval.metrics import classification_report, f1_score


if len(true_tags) != len(pred_tags):
    print(f"Warning: true tags and predicted tags have different lengths ({len(true_tags)} vs {len(pred_tags)}). Truncating to the shorter length.")
    min_len = min(len(true_tags), len(pred_tags))
    true_tags = true_tags[:min_len]
    pred_tags = pred_tags[:min_len]

print(classification_report([true_tags], [pred_tags]))
print("F1:", f1_score([true_tags], [pred_tags]))

In [ ]:
for t, p, true_token, pred_token in zip(true_tags, pred_tags, true_df.iloc[:, 0], pred_df.iloc[:, 0]):
    print(f"True: {t}, Pred: {p}, {true_token}, {pred_token}")